Week03_Sat

Máy hút bụi Model-based Reflex Agent sử dụng BFS và DFS có GUI được xây dựng bằng tkinter.

Link Github: https://github.com/BM910/AI-ARIN330585

Ma trận 2 chiều "room" biểu diễn sàn nhà:
    0 - Không có bụi, có thể đi qua được.
    1 - Có bụi, có thể đi qua được.
    2 - Vật cản, không đi qua được.
    3 - Máy hút bụi.

In [28]:
from collections import deque
import copy
import tkinter as tk
import random

current_room = None

class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

def find_roomba(state):
    for i in range(len(state)):
        for j in range(len(state[i])):
            if state[i][j] == 3 or state[i][j] == 4:
                return (i, j)
            
def get_moves(state):
    moves = []
    x, y = find_roomba(state)
    directions = [(-1, 0, "UP"), (1, 0, "DOWN"), (0, -1, "LEFT"), (0, 1, "RIGHT")]

    for dx, dy, action in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < len(state) and 0 <= ny < len(state[0]) and state[nx][ny] != 2:
            if state[nx][ny] == 1:
                action += f" and cleaned tile[{nx}, {ny}]"
            moves.append((dx, dy, action))

    return moves

def execute_move(state, direction):
    new_state = copy.deepcopy(state)
    x, y = find_roomba(new_state)
    dx, dy, action = direction
    new_state[x][y] = 0
    new_state[x + dx][y + dy] = 3
    return new_state

def is_clean(state):
    for row in state:
        for tile in row:
            if tile == 1:
                return False
    return True

def get_result_path(node):
    path = []
    while node.parent:
        path.append((node.state, node.action))
        node = node.parent
    return path[::-1]

def bfs1(room):
    if not current_room:
        return
    
    first_node = Node(room)

    if is_clean(first_node.state):
        return first_node

    frontier = deque()
    reached = set()
    frontier.append(first_node)
    max_frontier = 1

    while frontier:
        node = frontier.popleft()
        reached.add(str(node.state))
        moves = get_moves(node.state)

        if is_clean(node.state):
                update_log("BFS_1")
                update_log(f"frontier[] max size: {max_frontier}")
                update_log(f"reached() size: {len(reached)}")
                update_log("-"*20)
                return node
        
        for direction in moves:
            child_node = Node(execute_move(node.state, direction), node, direction[2])
            child_is_unique = True

            for frontier_node in frontier:
                if child_node.state == frontier_node.state:
                    child_is_unique = False
                    break

            if child_is_unique and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def bfs2(room):
    if not current_room:
        return
    
    first_node = Node(room)

    if is_clean(first_node.state):
        return first_node

    frontier = deque()
    reached = set()
    frontier.append(first_node)
    max_frontier = 1

    while frontier:
        node = frontier.popleft()
        reached.add(str(node.state))
        moves = get_moves(node.state)

        for direction in moves:
            child_node = Node(execute_move(node.state, direction), node, direction[2])
            if is_clean(child_node.state):
                update_log("BFS_2")
                update_log(f"frontier[] max size: {max_frontier}")
                update_log(f"reached() size: {len(reached)}")
                update_log("-"*20)
                return child_node
            child_is_unique = True

            for frontier_node in frontier:
                if child_node.state == frontier_node.state:
                    child_is_unique = False
                    break
                
            if child_is_unique and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def dfs1(room):
    if not current_room:
        return
    
    first_node = Node(room)

    if is_clean(first_node.state):
        return first_node

    frontier = deque()
    reached = set()
    frontier.append(first_node)
    max_frontier = 1

    while frontier:
        node = frontier.pop()
        reached.add(str(node.state))
        moves = get_moves(node.state)

        if is_clean(node.state):
                update_log("DFS_1")
                update_log(f"frontier[] max size: {max_frontier}")
                update_log(f"reached() size: {len(reached)}")
                update_log("-"*20)
                return node
        
        for direction in moves:
            child_node = Node(execute_move(node.state, direction), node, direction[2])
            if child_node not in frontier and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def dfs2(room):
    if not current_room:
        return
    
    first_node = Node(room)

    if is_clean(first_node.state):
        return first_node

    frontier = deque()
    reached = set()
    frontier.append(first_node)
    max_frontier = 1

    while frontier:
        node = frontier.pop()
        reached.add(str(node.state))
        moves = get_moves(node.state)

        for direction in moves:
            child_node = Node(execute_move(node.state, direction), node, direction[2])
            if is_clean(child_node.state):
                update_log("DFS_2")
                update_log(f"frontier[] max size: {max_frontier}")
                update_log(f"reached() size: {len(reached)}")
                update_log("-"*20)
                return child_node
            if child_node not in frontier and str(child_node.state) not in reached:
                frontier.append(child_node)
        max_frontier = max(max_frontier, len(frontier))

    return None

def randomize_map():
    global current_room

    rows = random.randint(3, 7)
    cols = rows

    choices = [0, 1, 2]
    weights = [0.60, 0.40, 0.20]

    current_room = []
    for r in range(rows):
        row_tiles = random.choices(choices, weights=weights, k=cols)
        current_room.append(row_tiles)
        
    roomba_placed = False
    while not roomba_placed:
        random_row = random.randint(0, rows - 1)
        random_col = random.randint(0, cols - 1)
        
        if current_room[random_row][random_col] in (0, 1):
            current_room[random_row][random_col] = 3
            roomba_placed = True

    draw_matrix_on_map(current_room)

def draw_matrix_on_map(matrix):
    # Clear anything previously drawn on the map canvas
    map_canvas.delete("all")
    
    rows = len(matrix)
    cols = len(matrix[0])
    
    # Dynamically scale cell size to perfectly fit your 400x400 map area
    cell_width = 400 // cols
    cell_height = 400 // rows
    
    # Map out the color scheme per rule
    colors = {
        0: "lightgray",  # Clean, passable floor
        1: "#7A592F",    # Dirty floor
        2: "#1B1A1A",    # Obstacle
        3: "green"     # Roomba
    }
    
    for r in range(rows):
        for c in range(cols):
            val = matrix[r][c]
            color = colors.get(val, "white")
            
            # Calculate pixel bounds for each grid coordinate
            x1 = c * cell_width
            y1 = r * cell_height
            x2 = x1 + cell_width
            y2 = y1 + cell_height
            
            # Draw the square onto the canvas
            map_canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="black", width=2)
            
            # Optional visual aid: Draw a distinct circle if it's the Roomba
            if val == 3:
                map_canvas.create_oval(x1+5, y1+5, x2-5, y2-5, fill="gray", outline="white", width=2)

def run_and_animate_algo(algo_func):
    global current_room
    if current_room is None:
        update_log("Please generate a map first using Randomize!")
        return

    final_node = algo_func(copy.deepcopy(current_room))
    
    if final_node is None:
        update_log("No solution found to clean the room!")
        return
        
    # path_timeline: [(state_1, action_1), (state_2, action_2), ...]
    path_timeline = get_result_path(final_node)
    
    animate_steps(path_timeline, 0)

def animate_steps(path_timeline, step_index):
    if step_index >= len(path_timeline):
        update_log("Animation complete! Room is clean.\n")
        log_path_timeline(path_timeline)
        return
    
    room_state, action_text = path_timeline[step_index]
    
    draw_matrix_on_map(room_state)
    
    for row in room_state:
        update_log(str(row))
    update_log("Roomba moved " + action_text)
    update_log("-"*20)

    window.after(100, lambda: animate_steps(path_timeline, step_index + 1))

def log_path_timeline(path_timeline):
    moves = ""
    for state, step in path_timeline:
        step = step.split(maxsplit=1)[0]
        moves += step + " -> "
    moves += "DONE\n"
    update_log(f"Path: {moves}")

def update_log(message="", clear=False):
    if clear:
        log_text.delete("1.0", tk.END)
    if message != "":
        log_text.insert(tk.END, message + "\n")
    
    log_text.see(tk.END)

# --- WINDOW SETUP ---
window = tk.Tk()
window.title("Layout Fix")

window.columnconfigure(0, weight=15)
window.columnconfigure(1, weight=40)
window.columnconfigure(2, weight=25)
window.rowconfigure(0, weight=1)
window.rowconfigure(1, weight=0)

# Sidebar Frame
frm_algo_options = tk.Frame(bg="gray", width=150, height=400, relief=tk.SUNKEN, border=5)
frm_algo_options.grid(column=0, row=0, sticky="nsew")
frm_algo_options.grid_propagate(False)
frm_algo_options.columnconfigure(0, weight=1)

# Map Frame (Replacing generic frame with a Canvas widget for pixel manipulation)
map_canvas = tk.Canvas(master=window, bg="black", width=400, height=400, relief=tk.SUNKEN, border=5, highlightthickness=0)
map_canvas.grid(column=1, row=0, sticky="nsew")
map_canvas.grid_propagate(False)

# Log Frame
frm_state_log = tk.Frame(bg="gray", width=250, height=400, relief=tk.SUNKEN, border=5)
frm_state_log.grid(column=2, row=0, sticky="nsew")
frm_state_log.grid_propagate(False)

# Actions Frame
frm_actions = tk.Frame(bg="darkgray", width=800, height=50, relief=tk.SUNKEN, border=5)
frm_actions.grid(column=0, columnspan=3, row=1, sticky="nsew")
frm_actions.grid_propagate(False)


# --- BUTTONS ---
btn_font = ("Helvetica", 12, "bold")

btn_bfs1 = tk.Button(frm_algo_options, text="BFS_1", bg="white", font=btn_font, command=lambda: run_and_animate_algo(bfs1))
btn_bfs1.grid(column=0, row=0, sticky="ew", padx=10, pady=5)

btn_bfs2 = tk.Button(frm_algo_options, text="BFS_2", bg="white", font=btn_font, command=lambda: run_and_animate_algo(bfs2))
btn_bfs2.grid(column=0, row=1, sticky="ew", padx=10, pady=5)

btn_dfs1 = tk.Button(frm_algo_options, text="DFS_1", bg="white", font=btn_font, command=lambda: run_and_animate_algo(dfs1))
btn_dfs1.grid(column=0, row=2, sticky="ew", padx=10, pady=5)

btn_dfs2 = tk.Button(frm_algo_options, text="DFS_2", bg="white", font=btn_font, command=lambda: run_and_animate_algo(dfs2))
btn_dfs2.grid(column=0, row=3, sticky="ew", padx=10, pady=5)

# Algorithm buttons and Randomize button spacing
frm_algo_options.rowconfigure(4, weight=1) 

btn_random = tk.Button(frm_algo_options, text="Randomize", bg="white", font=btn_font, command=randomize_map)
btn_random.grid(column=0, row=5, sticky="ew", padx=10, pady=10)

btn_clr_log = tk.Button(frm_algo_options, text="Clear log", bg="white", font=btn_font, command=lambda: update_log(clear=True))
btn_clr_log.grid(column=0, row=6, sticky="ew", padx=10, pady=10)

# --- LOG WIDGET SETUP ---
log_text = tk.Text(frm_state_log, bg="white", wrap=tk.WORD, font=("Consolas", 10))
log_scroll = tk.Scrollbar(frm_state_log, command=log_text.yview)
log_text.configure(yscrollcommand=log_scroll.set)

log_scroll.pack(side=tk.RIGHT, fill=tk.Y)
log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

window.mainloop()